## 消融实验

### 控制变量法验证每个模块的单独贡献

```
id_only -> id_text -> id_text_img -> full
  1         2           3            4
```

- 1→2: 文本语义增益 (BGE-large 1024d)
- 2→3: 视觉模态增益 (ViT-L/14 768d)
- 3→4: 跨域机制增益 (Sxy序列 + 交叉聚合)

异构特征: ID=512d (LightGCN) / Image=768d (ViT-L/14) / Text=1024d (BGE-large)

### 1. 环境 & 路径配置 & 超参

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, math, os, json, datetime
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

torch.backends.cudnn.benchmark = True

from cdsr_model import CDSRModel

# ==================== 文本变体选择 ====================
TEXT_VARIANT = "llmdesc"  # desc / llm / llmdesc

DATA_DIR = "final"
TEXT_FEAT_MAP = {
    "desc":    "text_bge_desc_1024.npy",
    "llm":     "text_bge_llm_1024.npy",
    "llmdesc": "text_bge_llmdesc_1024.npy",
}
TEXT_FEAT_PATH = os.path.join(DATA_DIR, "features", TEXT_FEAT_MAP[TEXT_VARIANT])
ID_FEAT_PATH   = os.path.join(DATA_DIR, "features", "item_id_lightgcn_512.npy")
IMG_FEAT_PATH  = os.path.join(DATA_DIR, "features", "image_features_768.npy")
TRAIN_CSV      = os.path.join(DATA_DIR, "train.csv")
VAL_CSV        = os.path.join(DATA_DIR, "val.csv")
TEST_CSV       = os.path.join(DATA_DIR, "test.csv")
ITEM2ID_PATH   = os.path.join(DATA_DIR, "item2id.json")
META_CSV       = os.path.join(DATA_DIR, "item_meta_merged.csv")

SAVE_DIR = os.path.join(DATA_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

RESULT_PATH = os.path.join(SAVE_DIR, "ablation_results.txt")

print(f"Text variant: {TEXT_VARIANT}")
print(f"特征路径: ID={ID_FEAT_PATH}")
print(f"          Img={IMG_FEAT_PATH}")
print(f"          Text={TEXT_FEAT_PATH}")

Text variant: llmdesc
特征路径: ID=final/features/item_id_lightgcn_512.npy
          Img=final/features/image_features_768.npy
          Text=final/features/text_bge_llmdesc_1024.npy



| 参数 | 值 | 说明 |
|------|:---:|------|
| `D_MODEL` | 256 | Transformer 隐层维度，所有异维特征 (512/768/1024) 投影到此统一空间 |
| `N_HEADS` | 4 | 多头注意力头数 |
| `N_LAYERS` | 2 | 自注意力层数，验证 2 层对序列推荐最优 |
| `DROPOUT` | 0.2 | 防止过拟合 |
| `MAX_SEQ_LEN` | 200 | 序列最大长度，截断过长历史 |
| `LR` | 1.4e-3 | AdamW 初始学习率 |
| `ABLATION_EPOCHS` | 5 | 每个变体训练轮数 |
| `LAMBDA1/LAMBDA2` | 0.3/0.1 | 跨域损失权重 |

In [2]:
D_MODEL      = 256
N_HEADS      = 4
N_LAYERS     = 2
DROPOUT      = 0.2
MAX_SEQ_LEN  = 200
LR           = 1.4e-3
ABLATION_EPOCHS = 5       # 消融每个变体训练轮数
PATIENCE     = 5
LAMBDA1      = 0.3
LAMBDA2      = 0.1
LABEL_SMOOTH = 0.0
USE_AMP      = True
NUM_WORKERS  = 8
EVAL_BS      = 256       # 评估用较小 BS，避免 score 矩阵 OOM
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# 消融变体列表: (ablation_mode, description)
ABLATION_VARIANTS = [
    ("id_only",      "纯ID (LightGCN 512d)"),
    ("id_text",      "+ BGE Text (LLM增强)"),
    ("id_text_img",  "+ ViT-L/14 Image"),
    ("full",         "+ 跨域机制"),
]

# 按变体大小 + 显存安全选择 BS（比旧版保守，Text=1024d 是旧版 2x）
VARIANT_BS = {
    "id_only":       768,   # ~30M, 512d emb
    "id_text":        512,   # ~65M, +1024d text emb
    "id_text_img":    384,   # ~98M, +768d img emb
    "full":           256,   # 126.3M, +cross-domain
}

print(f"Device: {DEVICE}  AMP: {USE_AMP}  Epochs: {ABLATION_EPOCHS}")
print(f"消融分支:")
for mode, desc in ABLATION_VARIANTS:
    print(f"  {mode:14s} (BS={VARIANT_BS[mode]:4d})  {desc}")

Device: cuda  AMP: True  Epochs: 5
消融分支:
  id_only        (BS= 768)  纯ID (LightGCN 512d)
  id_text        (BS= 512)  + BGE Text (LLM增强)
  id_text_img    (BS= 384)  + ViT-L/14 Image
  full           (BS= 256)  + 跨域机制


### 2. 加载特征 & 域掩码

域掩码从 item_meta_merged.csv 生成 (row index = item_id, domain 0=movie 1=book)

In [3]:
print("加载特征...")
id_feats = torch.from_numpy(np.load(ID_FEAT_PATH)).float()
img_feats = torch.from_numpy(np.load(IMG_FEAT_PATH)).float()
tex_feats = torch.from_numpy(np.load(TEXT_FEAT_PATH)).float()
print(f"ID:    {id_feats.shape}  (LightGCN)")
print(f"Image: {img_feats.shape}  (ViT-L/14)")
print(f"Text:  {tex_feats.shape}  (BGE-large, {TEXT_VARIANT})")

with open(ITEM2ID_PATH) as f:
    item2id = json.load(f)
N_ITEMS = len(item2id)
print(f"物品总数: {N_ITEMS}")
assert id_feats.shape[0] == img_feats.shape[0] == tex_feats.shape[0] == N_ITEMS

# 域掩码 (row index = item_id, domain=0=movie, domain=1=book)
meta_df = pd.read_csv(META_CSV)
movie_mask = torch.zeros(N_ITEMS, dtype=torch.bool)
book_mask  = torch.zeros(N_ITEMS, dtype=torch.bool)
movie_ids = meta_df[meta_df['domain'] == 0].index.tolist()
book_ids  = meta_df[meta_df['domain'] == 1].index.tolist()
movie_mask[movie_ids] = True
book_mask[book_ids]  = True
print(f"电影: {movie_mask.sum().item()}  图书: {book_mask.sum().item()}")

加载特征...
ID:    torch.Size([43528, 512])  (LightGCN)
Image: torch.Size([43528, 768])  (ViT-L/14)
Text:  torch.Size([43528, 1024])  (BGE-large, llmdesc)
物品总数: 43528
电影: 21280  图书: 22248


### 3. 构建用户序列

将原始交互记录（`user_id, item_id, domain, timestamp`）按用户分组并按时序排列，拆分为三个子序列：

- **SX**: 仅保留电影域交互 `(timestamp, item_id)`
- **SY**: 仅保留图书域交互
- **SXY**: 保留全部交互并按 `(timestamp, item_id, domain)` 三元组记录，用于跨域序列建模


In [4]:
def build_user_sequences(df):
    user_seqs = {}
    for uid, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")
        uid = int(uid)
        sx, sy, sxy = [], [], []
        for _, row in g.iterrows():
            ts = row["timestamp"]
            iid = int(row["item_id"])
            dom = int(row["domain"])
            if dom == 0:
                sx.append((ts, iid))
            else:
                sy.append((ts, iid))
            sxy.append((ts, iid, dom))
        user_seqs[uid] = {"sx": sx, "sy": sy, "sxy": sxy}
    return user_seqs

print("构建序列...")
train_seqs = build_user_sequences(pd.read_csv(TRAIN_CSV))
val_seqs   = build_user_sequences(pd.read_csv(VAL_CSV))
test_seqs  = build_user_sequences(pd.read_csv(TEST_CSV))

for name, seqs in [("Train", train_seqs), ("Val", val_seqs), ("Test", test_seqs)]:
    nu = len(seqs)
    nm = sum(len(s["sx"]) for s in seqs.values())
    nb = sum(len(s["sy"]) for s in seqs.values())
    print(f"{name}: {nu} users  movie={nm}  book={nb}")

构建序列...
Train: 20030 users  movie=376527  book=271423
Val: 20030 users  movie=9441  book=10589
Test: 20030 users  movie=8908  book=11122


### 4.生成训练/评估样本

将用户序列转换为训练样本，每个样本包含三个子序列的上下文和一个目标物品。

**样本类型**：

```
seq_type=0 (SX 样本):  context=SX[:i] + SY[before ts_i] + SXY[:p]
                        target=电影物品, loss → P_X
seq_type=1 (SY 样本):  context=SX[before ts_j] + SY[:j] + SXY[:q]
                        target=图书物品, loss → P_Y
seq_type=2 (SXY 样本): context=SX[before ts_k] + SY[before ts_k] + SXY[:k]
                        target=任意物品, loss → P_XY (full) / P_X or P_Y (消融)
```

`build_eval_samples` 是评估版本：用 train+val 的拼接序列做上下文，预测 val/test 中最后一个时间步的目标物品。

In [5]:
def build_samples(user_seqs):
    samples = []
    for uid, seqs in user_seqs.items():
        sx, sy, sxy = seqs["sx"], seqs["sy"], seqs["sxy"]
        if len(sxy) < 2:
            continue
        # SX+Y
        sx_ptr = sy_ptr = 0
        for k in range(1, len(sxy)):
            ts_k = sxy[k][0]
            while sx_ptr < len(sx) and sx[sx_ptr][0] < ts_k:
                sx_ptr += 1
            while sy_ptr < len(sy) and sy[sy_ptr][0] < ts_k:
                sy_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                                sy_ids=[y[1] for y in sy[:sy_ptr]],
                                sxy_ids=[z[1] for z in sxy[:k]],
                                target=sxy[k][1], target_domain=sxy[k][2], seq_type=2))
        # SX
        sx_to_sxy, sx_cnt = {}, 0
        for p, (ts, iid, dom) in enumerate(sxy):
            if dom == 0:
                sx_to_sxy[sx_cnt] = p; sx_cnt += 1
        sy_ptr = 0
        for i in range(1, len(sx)):
            while sy_ptr < len(sy) and sy[sy_ptr][0] < sx[i][0]:
                sy_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:i]],
                                sy_ids=[y[1] for y in sy[:sy_ptr]],
                                sxy_ids=[z[1] for z in sxy[:sx_to_sxy[i]]],
                                target=sx[i][1], target_domain=0, seq_type=0))
        # SY
        sy_to_sxy, sy_cnt = {}, 0
        for p, (ts, iid, dom) in enumerate(sxy):
            if dom == 1:
                sy_to_sxy[sy_cnt] = p; sy_cnt += 1
        sx_ptr = 0
        for j in range(1, len(sy)):
            while sx_ptr < len(sx) and sx[sx_ptr][0] < sy[j][0]:
                sx_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                                sy_ids=[y[1] for y in sy[:j]],
                                sxy_ids=[z[1] for z in sxy[:sy_to_sxy[j]]],
                                target=sy[j][1], target_domain=1, seq_type=1))
    return samples

def build_eval_samples(context_seqs, target_seqs):
    samples = []
    for uid, t_seqs in target_seqs.items():
        if uid not in context_seqs:
            continue
        ctx = context_seqs[uid]
        sx = ctx["sx"] + t_seqs["sx"]
        sy = ctx["sy"] + t_seqs["sy"]
        sxy = ctx["sxy"] + t_seqs["sxy"]
        if len(sxy) < 2:
            continue
        k = len(sxy) - 1
        ts_k = sxy[k][0]
        sx_ptr = len([x for x in sx if x[0] < ts_k])
        sy_ptr = len([y for y in sy if y[0] < ts_k])
        samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                            sy_ids=[y[1] for y in sy[:sy_ptr]],
                            sxy_ids=[z[1] for z in sxy[:k]],
                            target=sxy[k][1], target_domain=sxy[k][2]))
    return samples

print("构建训练样本...")
train_samples = build_samples(train_seqs)
print(f"Train: {len(train_samples)}")

val_samples = build_eval_samples(train_seqs, val_seqs)
print(f"Val: {len(val_samples)}")

train_val_seqs = {}
for uid in set(train_seqs.keys()) | set(val_seqs.keys()):
    ts = train_seqs.get(uid, {"sx": [], "sy": [], "sxy": []})
    vs = val_seqs.get(uid, {"sx": [], "sy": [], "sxy": []})
    train_val_seqs[uid] = {"sx": ts["sx"] + vs["sx"], "sy": ts["sy"] + vs["sy"], "sxy": ts["sxy"] + vs["sxy"]}

test_samples = build_eval_samples(train_val_seqs, test_seqs)
print(f"Test: {len(test_samples)}")

构建训练样本...
Train: 1235810
Val: 20030
Test: 20030


In [6]:
class CDSRDataset(Dataset):
    def __init__(self, samples, max_len=MAX_SEQ_LEN, is_train=True):
        self.samples = samples; self.max_len = max_len; self.is_train = is_train
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        def pad(ids):
            ids = ids[-self.max_len:]
            n = len(ids)
            return torch.tensor(ids + [0] * (self.max_len - n), dtype=torch.long), \
                   torch.tensor([1] * n + [0] * (self.max_len - n), dtype=torch.long)
        sx, sx_mask   = pad(s["sx_ids"])
        sy, sy_mask   = pad(s["sy_ids"])
        sxy, sxy_mask = pad(s["sxy_ids"])
        if self.is_train:
            return (sx, sx_mask, sy, sy_mask, sxy, sxy_mask,
                    torch.tensor(s["target"], dtype=torch.long),
                    torch.tensor(s["target_domain"], dtype=torch.long),
                    torch.tensor(s["seq_type"], dtype=torch.long))
        else:
            return (sx, sx_mask, sy, sy_mask, sxy, sxy_mask,
                    torch.tensor(s["target"], dtype=torch.long),
                    torch.tensor(s["target_domain"], dtype=torch.long))

def collate_train(batch):
    sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom, seq_type = zip(*batch)
    return (torch.stack(list(sx)), torch.stack(list(sx_mask)),
            torch.stack(list(sy)), torch.stack(list(sy_mask)),
            torch.stack(list(sxy)), torch.stack(list(sxy_mask)),
            torch.stack(list(target)), torch.stack(list(tgt_dom)),
            torch.stack(list(seq_type)))

def collate_eval(batch):
    sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom = zip(*batch)
    return (torch.stack(list(sx)), torch.stack(list(sx_mask)),
            torch.stack(list(sy)), torch.stack(list(sy_mask)),
            torch.stack(list(sxy)), torch.stack(list(sxy_mask)),
            torch.stack(list(target)), torch.stack(list(tgt_dom)))

train_ds = CDSRDataset(train_samples, is_train=True)
val_ds   = CDSRDataset(val_samples,   is_train=False)
test_ds  = CDSRDataset(test_samples,  is_train=False)

NUM_WORKERS_ACTUAL = min(NUM_WORKERS, os.cpu_count() or 4)
# 评估用较小 BS，避免 score 矩阵 (batch × 43528) OOM
val_loader   = DataLoader(val_ds, batch_size=EVAL_BS, shuffle=False,
                           collate_fn=collate_eval, num_workers=NUM_WORKERS_ACTUAL, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=EVAL_BS, shuffle=False,
                           collate_fn=collate_eval, num_workers=NUM_WORKERS_ACTUAL, pin_memory=True)

print(f"Eval BS={EVAL_BS}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

Eval BS=256  Val batches: 79  Test batches: 79


### 5. 训练 & 评估函数

### 损失设计
- **单域变体**: L = CE(P_X, target[dom=0]) + lambda1 * CE(P_Y, target[dom=1])
- **Full 变体**: L = L_X + lambda1 * L_Y + lambda2 * L_XY

### 评估
每个 epoch 后：Val + Test 全量排序 HR@10, NDCG@10, MRR (含分域)

### 保存
- `abl_{mode}_best.pt` — 最优模型权重 (按 loss)
- `abl_{mode}_best_val.pt` — 最优模型权重 (按 Val HR@10)
- `abl_{mode}_ckpt.pt` — 断点文件 (可续训)

In [7]:
def train_one_variant(ablation_mode, description):
    """训练单个消融变体，返回评估指标。"""
    print(f"\n{'='*60}")
    print(f"  消融变体: {description}  (mode={ablation_mode})")
    print(f"{'='*60}")

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    model = CDSRModel(N_ITEMS, id_feats, img_feats, tex_feats,
                       movie_mask, book_mask,
                       d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                       dropout=DROPOUT, max_len=MAX_SEQ_LEN,
                       ablation=ablation_mode).to(DEVICE)

    n_p = sum(p.numel() for p in model.parameters())
    print(f"参数量: {n_p/1e6:.1f}M  modalities={model.enabled_modalities}  cross_domain={model.cross_domain}")

    bs = VARIANT_BS[ablation_mode]
    variant_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                                 collate_fn=collate_train, num_workers=NUM_WORKERS_ACTUAL,
                                 pin_memory=True)
    print(f"Batch size: {bs}  batches/epoch: {len(variant_loader)}")

    scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01,
                                   fused=True if DEVICE == "cuda" else False)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

    @torch.no_grad()
    def evaluate(loader, k=10):
        model.eval()
        hits, ndcg_sum, mrr_sum, total = 0, 0, 0, 0
        dh, dt = {0: 0, 1: 0}, {0: 0, 1: 0}
        for sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom in loader:
            sx, sx_mask = sx.to(DEVICE), sx_mask.to(DEVICE)
            sy, sy_mask = sy.to(DEVICE), sy_mask.to(DEVICE)
            sxy, sxy_mask = sxy.to(DEVICE), sxy_mask.to(DEVICE)
            target = target.to(DEVICE); tgt_dom = tgt_dom.to(DEVICE)
            out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=True)
            for i in range(len(target)):
                dom = tgt_dom[i].item()
                if dom == 0:
                    score = out["P_X"][i]
                    if "P_Y_to_X" in out:
                        score = score + LAMBDA1 * out["P_Y_to_X"][i]
                    if "P_XY_to_X" in out:
                        score = score + LAMBDA2 * out["P_XY_to_X"][i]
                else:
                    score = out["P_X_to_Y"][i] if "P_X_to_Y" in out else 0
                    if "P_Y" in out:
                        score = score + LAMBDA1 * out["P_Y"][i]
                    if "P_XY_to_Y" in out:
                        score = score + LAMBDA2 * out["P_XY_to_Y"][i]
                _, topk = torch.topk(score, k)
                topk = topk.cpu().tolist()
                tgt = target[i].item()
                if tgt in topk:
                    rank = topk.index(tgt) + 1
                    hits += 1; dh[dom] += 1
                    ndcg_sum += 1.0 / math.log2(rank + 1)
                    mrr_sum += 1.0 / rank
                total += 1; dt[dom] += 1
        hr = hits / total if total else 0
        ndcg = ndcg_sum / total if total else 0
        mrr = mrr_sum / total if total else 0
        return hr, ndcg, mrr, total, dh, dt

    # 断点续训
    best_path = os.path.join(SAVE_DIR, f"abl_{ablation_mode}_best.pt")
    ckpt_path = os.path.join(SAVE_DIR, f"abl_{ablation_mode}_ckpt.pt")
    start_epoch = 1
    best_val_hr = 0.0
    best_val_mrr = 0.0
    no_improve = 0
    has_cross = model.cross_domain

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if scaler is not None and ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_hr = ckpt.get("best_val_hr", 0.0)
        best_val_mrr = ckpt.get("best_val_mrr", 0.0)
        no_improve = ckpt["no_improve"]
        print(f"  从断点恢复: epoch={start_epoch}, best_val_hr={best_val_hr:.4f}")
    else:
        print("  从头开始训练")

    print(f"  {'Epoch':>6} {'TrainLoss':>10} {'ValHR@10':>9} {'ValNDCG':>8} {'ValMRR':>8} "
          f"{'TestHR@10':>10} {'TestNDCG':>9} {'TestMRR':>9} {'MovieHR':>8} {'BookHR':>8}")
    print("  " + "-" * 95)

    for epoch in range(start_epoch, ABLATION_EPOCHS + 1):
        model.train()
        total_loss, n_batch = 0, 0
        pbar = tqdm(variant_loader, desc=f"  [{ablation_mode}] Epoch {epoch}/{ABLATION_EPOCHS}")

        for sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom, seq_type in pbar:
            sx, sx_mask = sx.to(DEVICE), sx_mask.to(DEVICE)
            sy, sy_mask = sy.to(DEVICE), sy_mask.to(DEVICE)
            sxy, sxy_mask = sxy.to(DEVICE), sxy_mask.to(DEVICE)
            target, tgt_dom, seq_type = target.to(DEVICE), tgt_dom.to(DEVICE), seq_type.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=False)
                loss = torch.tensor(0.0, device=DEVICE)
                if has_cross:
                    for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y"), (2, LAMBDA2, "P_XY")]:
                        m = (seq_type == st)
                        if m.any():
                            loss = loss + wt * criterion(out[key][m], target[m])
                else:
                    for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y")]:
                        m = (seq_type == st)
                        if m.any():
                            loss = loss + wt * criterion(out[key][m], target[m])
                    m2 = (seq_type == 2)
                    if m2.any():
                        for dom_val, dom_key, dom_wt in [(0, "P_X", 1.0), (1, "P_Y", LAMBDA1)]:
                            dm = m2 & (tgt_dom == dom_val)
                            if dm.any():
                                loss = loss + LAMBDA2 * dom_wt * criterion(out[dom_key][dm], target[dm])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=False)
                loss = torch.tensor(0.0, device=DEVICE)
                if has_cross:
                    for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y"), (2, LAMBDA2, "P_XY")]:
                        m = (seq_type == st)
                        if m.any():
                            loss = loss + wt * criterion(out[key][m], target[m])
                else:
                    for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y")]:
                        m = (seq_type == st)
                        if m.any():
                            loss = loss + wt * criterion(out[key][m], target[m])
                    m2 = (seq_type == 2)
                    if m2.any():
                        for dom_val, dom_key, dom_wt in [(0, "P_X", 1.0), (1, "P_Y", LAMBDA1)]:
                            dm = m2 & (tgt_dom == dom_val)
                            if dm.any():
                                loss = loss + LAMBDA2 * dom_wt * criterion(out[dom_key][dm], target[dm])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item(); n_batch += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / n_batch

        val_hr, val_ndcg, val_mrr, _, val_dh, val_dt = evaluate(val_loader)
        test_hr, test_ndcg, test_mrr, _, test_dh, test_dt = evaluate(test_loader)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        movie_hr = test_dh[0] / test_dt[0] if test_dt[0] else 0
        book_hr = test_dh[1] / test_dt[1] if test_dt[1] else 0

        print(f"  {epoch:>6} {avg_loss:>10.4f} {val_hr:>9.4f} {val_ndcg:>8.4f} {val_mrr:>8.4f} "
              f"{test_hr:>10.4f} {test_ndcg:>9.4f} {test_mrr:>9.4f} {movie_hr:>8.4f} {book_hr:>8.4f}")
        scheduler.step(val_hr)

        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                       scheduler=scheduler.state_dict(),
                       scaler=scaler.state_dict() if scaler else {},
                       epoch=epoch, best_val_hr=best_val_hr, best_val_mrr=best_val_mrr,
                       no_improve=no_improve), ckpt_path)

        if val_hr > best_val_hr + 1e-5:
            best_val_hr = val_hr; no_improve = 0
            torch.save(model.state_dict(), best_path)
            print("  -> saved (best hr)")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  Early stop at epoch {epoch}"); break

        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"abl_{ablation_mode}_best_val.pt"))

    # 最终评估（加载最优 HR 模型）
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()
    eval_val_hr, eval_val_ndcg, eval_val_mrr, _, _, _ = evaluate(val_loader)
    eval_test_hr, eval_test_ndcg, eval_test_mrr, _, test_dh, test_dt = evaluate(test_loader)

    print(f"  Final (best_hr): Val HR={eval_val_hr:.4f} NDCG={eval_val_ndcg:.4f} MRR={eval_val_mrr:.4f}  "
          f"Test HR={eval_test_hr:.4f} NDCG={eval_test_ndcg:.4f} MRR={eval_test_mrr:.4f}")

    # 先保存属性再清理模型
    saved_modalities = list(model.enabled_modalities)
    saved_cross = model.cross_domain

    del variant_loader, model, optimizer, scheduler
    if scaler is not None: del scaler
    if DEVICE == "cuda": torch.cuda.empty_cache()

    return dict(mode=ablation_mode, desc=description, params_M=n_p/1e6,
                val_hr=eval_val_hr, val_ndcg=eval_val_ndcg, val_mrr=eval_val_mrr,
                test_hr=eval_test_hr, test_ndcg=eval_test_ndcg, test_mrr=eval_test_mrr,
                test_movie_hr=test_dh[0]/test_dt[0] if test_dt[0] else 0,
                test_book_hr=test_dh[1]/test_dt[1] if test_dt[1] else 0,
                modalities=saved_modalities,
                cross_domain=saved_cross)

### 6. 主循环 — 依次训练 4 个变体

In [ ]:
print(f"\n消融实验开始: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Text variant: {TEXT_VARIANT}")

results = []
for mode, desc in ABLATION_VARIANTS:
    r = train_one_variant(mode, desc)
    results.append(r)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"\n--- 已完成 {len(results)}/{len(ABLATION_VARIANTS)} ---")
    for r_ in results:
        print(f"  {r_['mode']:14s}: Val HR={r_['val_hr']:.4f}  Test HR={r_['test_hr']:.4f}  MRR={r_['test_mrr']:.4f}")

print(f"\n消融实验完成: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
print("\n" + "=" * 105)
print("  消融实验结果汇总")
print("=" * 105)
print(f"{'Branch':<32} {'Params':>7} {'ValHR':>7} {'ValNDCG':>8} {'ValMRR':>8} "
      f"{'TestHR':>8} {'TestNDCG':>9} {'TestMRR':>9} {'MovieHR':>8} {'BookHR':>8} {'deltaHR':>8}")
print("-" * 105)

baseline_hr = results[0]["test_hr"]
for i, r in enumerate(results):
    label = f"{r['mode']} ({r['desc']})"
    delta = r["test_hr"] - baseline_hr if i > 0 else 0
    delta_str = f"+{delta:.4f}" if delta > 0 else f"{delta:.4f}"
    print(f"{label:<32} {r['params_M']:>6.1f}M {r['val_hr']:>7.4f} {r['val_ndcg']:>8.4f} {r['val_mrr']:>8.4f} "
          f"{r['test_hr']:>8.4f} {r['test_ndcg']:>9.4f} {r['test_mrr']:>9.4f} "
          f"{r['test_movie_hr']:>8.4f} {r['test_book_hr']:>8.4f} {delta_str:>8}")

print("-" * 105)

# 增量贡献
print("\n增量贡献 (每次只加一个模块):")
print(f"{'Step':<35} {'HR@10':>8} {'NDCG@10':>9} {'MRR':>9}")
for i in range(1, len(results)):
    prev, curr = results[i-1], results[i]
    gain_hr = curr["test_hr"] - prev["test_hr"]
    gain_ndcg = curr["test_ndcg"] - prev["test_ndcg"]
    gain_mrr = curr["test_mrr"] - prev["test_mrr"]
    print(f"  + {curr['desc']:<30s}: {gain_hr:>+8.4f} {gain_ndcg:>+9.4f} {gain_mrr:>+9.4f}")

# 总增益
total_gain_hr = results[-1]["test_hr"] - results[0]["test_hr"]
total_gain_mrr = results[-1]["test_mrr"] - results[0]["test_mrr"]
print(f"\n  总增益 (full - id_only):              HR@10={total_gain_hr:+.4f}  MRR={total_gain_mrr:+.4f}")

# 保存结果
with open(RESULT_PATH, "w", encoding="utf-8") as f:
    f.write(f"消融实验结果 — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Text variant: {TEXT_VARIANT}\n")
    f.write("=" * 70 + "\n")
    for r in results:
        f.write(f"{r['mode']} ({r['desc']}): "
                f"ValHR={r['val_hr']:.4f} ValNDCG={r['val_ndcg']:.4f} ValMRR={r['val_mrr']:.4f}  "
                f"TestHR={r['test_hr']:.4f} TestNDCG={r['test_ndcg']:.4f} TestMRR={r['test_mrr']:.4f}  "
                f"Params={r['params_M']:.1f}M  modalities={r['modalities']}  "
                f"cross_domain={r['cross_domain']}\n")
    f.write("\n增量贡献:\n")
    for i in range(1, len(results)):
        gain_hr = results[i]["test_hr"] - results[i-1]["test_hr"]
        gain_mrr = results[i]["test_mrr"] - results[i-1]["test_mrr"]
        f.write(f"  + {results[i]['desc']}: HR@10 {gain_hr:+.4f}  MRR {gain_mrr:+.4f}\n")
    f.write(f"\n总增益: HR@10={total_gain_hr:+.4f}  MRR={total_gain_mrr:+.4f}\n")

print(f"\n结果已保存至 {RESULT_PATH}")
print("\n消融实验全部完成。")